# Cyclistic Bike-Share: Monthly Data Validation

This notebook documents the Python validation workflow used to assess data quality across 12 monthly Divvy bike-share datasets (October 2025–September 2026).

The workflow checks for missing values, duplicate records, inconsistent data, and potential anomalies before the datasets are combined and cleaned for analysis.

In [1]:
import pandas as pd

## Validation functions

The following functions perform reusable data-quality checks on each monthly dataset. They calculate ride durations and identify duplicate IDs, missing values, inconsistent station information, and other potential anomalies.

In [2]:
def calculate_ride_lengths(rides):
    rides = rides.copy()
    rides['started_at'] = pd.to_datetime(rides['started_at'])
    rides['ended_at'] = pd.to_datetime(rides['ended_at'])
    rides['ride_length_sec'] = ( 
    rides['ended_at'] - rides['started_at']).dt.total_seconds()
    return rides

In [3]:
def count_invalid_durations(rides):
    return (rides['ride_length_sec'] <= 0).sum()

In [4]:
def count_duplicate_ids(rides):
    return rides['ride_id'].duplicated().sum()

In [5]:
def count_missing_values(rides):
    return rides.isna().sum()

In [6]:
def count_short_rides(rides):
    return ((rides['ride_length_sec'] > 0) & (rides['ride_length_sec'] < 60)).sum()

In [7]:
def count_start_station_mismatches(rides):
    name_missing = rides['start_station_name'].isna()
    id_missing = rides['start_station_id'].isna()
    return (name_missing != id_missing).sum()

In [8]:
def count_end_station_mismatches(rides):
    name_missing = rides['end_station_name'].isna()
    id_missing = rides['end_station_id'].isna()
    return (name_missing != id_missing).sum()

In [9]:
expected_columns = ['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual']

In [10]:
def check_columns(rides):
    return set(rides.columns) == set(expected_columns)

In [11]:
expected_membership = {'casual', 'member'}
expected_bike_type = {'electric_bike', 'classic_bike'}

In [12]:
def count_unexpected_values(rides, column, expected_values):
    return (~rides[column].isin(expected_values)).sum()

In [13]:
def count_invalid_timestamps(rides, column):
    return pd.to_datetime(rides[column], errors='coerce').isna().sum()

In [14]:
def count_out_of_month_starts(rides, expected_month):
    return (rides['started_at'].dt.to_period('M') != expected_month).sum()

## Monthly data files

The workflow uses Python's pathlib library to locate the monthly CSV files. The files are collected and sorted before validation to ensure consistent processing order.

In [15]:
from pathlib import Path

In [17]:
data_folder = Path("../Cyclistic_data_raw")

In [19]:
csv_files = list(data_folder.glob('*.csv'))

In [20]:
csv_files = sorted(csv_files)

## Automated monthly validation

The following loop processes each monthly CSV file, applies the validation functions defined above, and collects the results into a structured summary. This allows the same data-quality checks to be performed consistently across all 12 datasets.

In [21]:
all_results = []

for file in csv_files:
    rides = pd.read_csv(file)
    month = f'{file.stem[:4]}-{file.stem[4:6]}'
    prepared = calculate_ride_lengths(rides)

    results = {
        'month': month, 
        'total_rides': len(rides), 
        'duplicate_ids': 
    int(count_duplicate_ids(rides)), 
        'missing_values_total': 
    int(count_missing_values(rides).sum()),
        'columns_valid': check_columns(rides),
        'unexpected_membership': 
    int(count_unexpected_values(rides, 'member_casual', expected_membership)),
        'unexpected_bike_types': 
    int(count_unexpected_values(rides, 'rideable_type', expected_bike_type)),
        'invalid_start_timestamps': 
    int(count_invalid_timestamps(rides, 'started_at')),
        'invalid_end_timestamps':
    int(count_invalid_timestamps(rides, 'ended_at')),
        'start_station_mismatches':
    int(count_start_station_mismatches(rides)),
        'end_station_mismatches':
    int(count_end_station_mismatches(rides)),
        'invalid_durations':
    int(count_invalid_durations(prepared)),
        'short_rides':
    int(count_short_rides(prepared)),
        'out_of_month_starts':
    int(count_out_of_month_starts(prepared, month))
            
        
    }
    all_results.append(results)

## Validation summary

The results from all 12 monthly datasets are combined into a summary DataFrame and exported to CSV for review. This provides a consolidated record of the data-quality checks performed across the full analysis period.

In [22]:
validation_summary = pd.DataFrame(all_results)
validation_summary

,month,total_rides,duplicate_ids,missing_values_total,columns_valid,unexpected_membership,unexpected_bike_types,invalid_start_timestamps,invalid_end_timestamps,start_station_mismatches,end_station_mismatches,invalid_durations,short_rides,out_of_month_starts
0,2025-10,646039,0,598062,True,0,0,0,0,0,0,0,17016,71
1,2025-11,356628,0,317066,True,0,0,0,0,0,0,29,9520,128
2,2025-12,140534,0,114186,True,0,0,0,0,0,0,0,3961,12
3,2026-01,137787,0,104688,True,0,0,0,0,0,0,0,3846,23
4,2026-02,201450,0,134118,True,0,0,0,0,0,0,0,4022,18
5,2026-03,317037,0,241632,True,0,0,0,0,0,0,0,8286,22
6,2026-04,448252,0,373270,True,0,0,0,0,0,0,0,12420,0
7,2026-05,653704,0,559558,True,0,0,0,0,0,0,0,17769,35
8,2026-06,762550,0,647132,True,0,0,0,0,0,0,0,20516,78
9,2026-07,869051,0,773240,True,0,0,0,0,0,0,0,22315,123


In [ ]:
validation_summary.to_csv('validation_summary.csv', index=False)